# Vinamilk OCR v9 paired CPU benchmark

Run Tesseract, PaddleOCR, EasyOCR, or Surya on the same 14 reviewed regions from six Vinamilk PDFs at 180 DPI. This notebook is deliberately CPU-only: attach the private dataset `ltrngnhn/vinamilk-ocr-v9-benchmark-assets`, keep Kaggle GPU disabled, and reject any CUDA-enabled runtime. Run one engine per kernel, preserving the exact v9 label and source/render hashes for paired comparison.

The private dataset contains evaluator code, reviewed references, and six source PDFs; it contains no credentials or user-uploaded documents. Each kernel records engine version, device, CPU architecture/count, page latency, memory and image hashes; the runtime CPU model is recorded when Kaggle exposes it. The Surya option pins version 0.17.1 (Surya v1). This run improves hardware comparability but remains diagnostic: reference quality is mixed, and it is not model-selection evidence.


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

input_root = Path("/kaggle/input")
for bundle in [*input_root.glob("*/benchmark_project.zip"), *input_root.glob("datasets/*/*/benchmark_project.zip")]:
    extracted = Path("/kaggle/working/benchmark-assets/project")
    shutil.rmtree(extracted.parent, ignore_errors=True)
    shutil.unpack_archive(str(bundle), str(extracted.parent))
project_candidates = {
    *input_root.glob("*/project"),
    *input_root.glob("datasets/*/*/project"),
    *Path("/kaggle/working/benchmark-assets").glob("project"),
    *(entry.parent.parent for entry in input_root.rglob("scripts.processing.evaluate_ocr.py") if entry.parent.name == "scripts"),
}
assets_candidates = sorted(
    project for project in project_candidates
    if (project / "scripts" / "scripts.processing.evaluate_ocr.py").is_file()
)
if len(assets_candidates) != 1:
    raise FileNotFoundError(
        "Expected exactly one attached OCR benchmark assets/project; "
        f"found: {assets_candidates}"
    )
assets = assets_candidates[0]
repo = Path("/kaggle/working/enterprise-knowledge-ai-agent")
if repo.exists():
    shutil.rmtree(repo)
shutil.copytree(assets, repo)
%cd /kaggle/working/enterprise-knowledge-ai-agent


In [ ]:
ENGINE = "tesseract"  # Change to paddleocr, easyocr, or surya for separate kernels.
DPI = 180
TESSERACT_PSM = 3  # Use 11 for the paired sparse-text layout comparison.
DATASET_PATH = "data/evaluation/vinamilk_ocr_ground_truth_v9.json"
DATASET_TAG = "v9"


In [ ]:
import subprocess, sys

# Install one engine only, avoiding incompatible OCR framework combinations.
subprocess.run([sys.executable, "-m", "pip", "install", "PyMuPDF==1.27.2.3", "Pillow==12.1.0"], check=True)
if ENGINE == "paddleocr":
    subprocess.run([sys.executable, "-m", "pip", "--default-timeout=300", "--retries=5", "install", "paddlepaddle==3.3.0", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cpu/"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "paddleocr==3.7.0"], check=True)
elif ENGINE == "easyocr":
    subprocess.run([sys.executable, "-m", "pip", "install", "easyocr==1.7.2"], check=True)
elif ENGINE == "surya":
    subprocess.run([sys.executable, "-m", "pip", "install", "surya-ocr==0.17.1", "transformers==4.57.3"], check=True)
elif ENGINE == "tesseract":
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-vie"], check=True)
else:
    raise ValueError(f"Unknown ENGINE: {ENGINE}")


In [ ]:
import os

os.environ["HF_HOME"] = "/tmp/ocr-model-cache/huggingface"
os.environ["TORCH_HOME"] = "/tmp/ocr-model-cache/torch"
if ENGINE == "surya":
    # Limit per-page CPU inference memory for this paired run.
    os.environ["DETECTOR_BATCH_SIZE"] = "4"
    os.environ["RECOGNITION_BATCH_SIZE"] = "16"
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
import torch
cuda_available = torch.cuda.is_available()
print("torch", torch.__version__, "CUDA", cuda_available)
if cuda_available:
    raise RuntimeError("Controlled v9 comparison requires a CPU-only Kaggle runtime.")
if ENGINE == "paddleocr":
    import paddle
    paddle_device = paddle.device.get_device()
    print("paddle", paddle.__version__, "CUDA build", paddle.is_compiled_with_cuda(), "device", paddle_device)
    if paddle_device != "cpu":
        raise RuntimeError("Controlled v9 comparison requires PaddleOCR CPU execution.")


In [ ]:
import json, os, platform
dataset_path = Path(DATASET_PATH)
if not dataset_path.is_file():
    raise FileNotFoundError(f"Missing OCR labels: {dataset_path}")
dataset = json.loads(dataset_path.read_text(encoding="utf-8"))
for source in sorted({page["file"] for page in dataset["pages"]}):
    if not Path(source).is_file():
        raise FileNotFoundError(f"Missing benchmark source PDF: {source}")
output = Path("/kaggle/working") / f"vinamilk-ocr-{DATASET_TAG}-{ENGINE}.json"
command = [sys.executable, "scripts/processing/evaluate_ocr.py", "--dataset", DATASET_PATH, "--engines", ENGINE, "--dpi", str(DPI), "--output", str(output)]
if ENGINE == "tesseract":
    command.extend(["--tesseract-psm", str(TESSERACT_PSM)])
subprocess.run(command, check=True)
report = json.loads(output.read_text(encoding="utf-8"))
cpu_model = None
if Path("/proc/cpuinfo").is_file():
    for line in Path("/proc/cpuinfo").read_text(errors="replace").splitlines():
        if line.lower().startswith(("model name", "hardware")):
            cpu_model = line.split(":", 1)[-1].strip()
            if cpu_model:
                break
if not cpu_model:
    candidate = platform.processor()
    cpu_model = candidate if candidate and candidate != platform.machine() else None
report["environment"]["cpu_model"] = cpu_model or "not exposed by runtime"
report["environment"]["cpu_architecture"] = platform.machine()
report["environment"]["logical_cpus"] = os.cpu_count()
output.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Download the result from Kaggle Outputs: {output}")


## Limitations

This paired run uses CPU-only Kaggle kernels for all four OCR engines, with matching v9 labels, source PDFs, 180-DPI renders and Tesseract PSM 3. CPU model/count are recorded so runtime comparisons can be checked; kernel-to-kernel host variation can still affect throughput. OCR table row/cell scores are text-linearized and do not establish spatial reconstruction. The v9 references mix manually transcribed and text-layer-assisted material and remain too small for model selection.